In [23]:
import os
#os.chdir("../")

In [24]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainingConfig:
    root_dir: Path
    train_data: Path
    test_data: Path
    model_name: str
    target_column: str
    alpha: float
    l1_ratio: float

In [25]:
from src.ds_edep.utils.common import read_yaml, main_logger, create_directories
from src.ds_edep.constants import *

In [ ]:
class ConfigManager:
    def __init__(self,
                 config_path = CONFIG_FILE_PATH,
                 params_path = PARAMS_FILE_PATH,
                 schema_path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)
        self.schema = read_yaml(schema_path)
        
        create_directories([self.config.artifacts_root])
        
    def get_model_trainer_config(self):
        config = self.config.model_training
        params = self.params.model_param
        schema = self.schema
        
        obj_config = ModelTrainingConfig(
            root_dir = config.root_dir,
            train_data = config.train_data_path,
            test_data = config.test_data_path,
            model_name = config.model_name,
            target_column = schema.TARGET_COLUMN.name,
            alpha = params.alpha,
            l1_ratio = params.l1_ratio
        )
        
        return obj_config
        

In [27]:
import pandas as pd

In [28]:
from sklearn.linear_model import ElasticNet
import joblib

In [ ]:
class ModelTrainer:
    def __init__(self, config):
        self.config = config
        
    def train(self):
        train_data = pd.read_csv(self.config.train_data)
        
        X_train = train_data.drop(columns=[self.config.target_column])
        y_train = train_data[self.config.target_column]
        
        model = ElasticNet(l1_ratio= self.config.l1_ratio, alpha=self.config.alpha)
        model.fit(X_train, y_train)

        joblib.dump(model, os.path.join(self.config.root_dir, self.config.model_name))
        

In [30]:
try:
    config = ConfigManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-06-08 11:23:08,716: INFO: common: created directory at: artifacts]


BoxKeyError: "'ConfigBox' object has no attribute 'str'"